# Model and embedding details

Use this notebook as an architecture reference during the demo. It explains the embedding dimensions, Transformer settings, encoder-only design, supervised learning setup, and output layer used in the compact Colab demo.

The model configuration here is intentionally small and CPU-friendly. The goal is to demonstrate the multimodal representation pipeline, not to train a production-scale trading model.

## 1. Demo-level configuration

| Component | Demo value | Meaning |
|---|---:|---|
| Rolling window | `20` | each tabular sample uses 20 historical rows |
| Tabular feature count | `11` | engineered OHLCV / benchmark-relative features |
| Text embedding dimension | `16` | deterministic lightweight text vector |
| Image embedding dimension | `16` | output of the demo image Transformer encoder |
| KG embedding dimension | usually `4` | peer count, peer return, sector return, event flag; inspect NPZ for exact width |
| Shared fusion dimension | `16` | common Transformer dimension after modality projections |
| Fusion attention heads | `4` | multi-head self-attention heads |
| Fusion layers | `1` | one Transformer encoder layer in the demo run |
| Fusion feed-forward dimension | `32` | inner MLP dimension in the Transformer block |

The all-in-one notebook prints the actual tensor shapes after it builds `real_world_multimodal_samples.npz`. Those printed shapes are the source of truth for a particular run.

## 2. Architectural note: encoder-only Transformer

This project uses an **encoder-style Transformer**, not an encoder-decoder Transformer.

The task is binary prediction/classification:

> Given tabular, image, text, and KG signals for a stock/date sample, predict whether the stock will outperform the Nifty benchmark over the configured horizon.

Because the model emits one prediction rather than generating a sequence, a Transformer decoder is not needed.

Current architecture:

```text
modality tokens
  -> projection to shared dimension
  -> TransformerEncoder self-attention
  -> pooled fused representation
  -> dropout
  -> Linear(model_dim -> 1)
  -> binary logit
  -> sigmoid(logit) at inference
```

A decoder would make sense in a future extension if the system needed to generate text, such as an analyst-style explanation or natural-language rationale. That is not part of the current demo model.

## 3. How the model learns

The model learns from **supervised training samples with labels**. Self-attention is the mechanism that mixes information across tokens and modalities; it is not the supervision signal by itself.

One training sample is conceptually:

```text
stock_id = RELIANCE.NS
end_date = 2025-12-01

tabular_tokens = last 20 rows of engineered market features
image_tokens   = candlestick chart embedding for the same date window
text_tokens    = text/context embedding available up to that date
kg_tokens      = sector/peer/event context up to that date

y = 1 if RELIANCE.NS outperformed Nifty over the next horizon
y = 0 otherwise
```

So the training row is:

```text
(tabular_tokens, image_tokens, text_tokens, kg_tokens) -> y
```

The supervised learning loop is:

```text
1. Build labelled samples
2. Pass aligned modality tokens through the model
3. Model outputs one raw score: logit
4. Compare logit with true label y
5. Compute binary cross-entropy loss
6. Backpropagate the loss
7. Update model weights
```

In code-like form:

```python
prediction_logit = model(tabular, image, text, kg)
loss = BCEWithLogitsLoss(prediction_logit, true_label)
loss.backward()
optimizer.step()
```

Self-attention helps the model decide which signals should influence each other, for example whether a tabular trend token should attend to a chart token, or whether sector context should reinforce a price/volume signal. But the model learns what is useful only because the prediction is compared with the actual future outperformance label.

Key distinction:

```text
Training samples + labels = what teaches the model
Self-attention           = how the model combines signals
Loss function            = how the model knows it was wrong
Gradient descent         = how the model updates itself
```

## 4. Modality-by-modality details

### Tabular modality

**Raw input:** OHLCV and benchmark data from yfinance.

**Feature set in the demo:**

```text
log_return_1d
cum_return_3d
cum_return_5d
cum_return_10d
realized_vol_5d
realized_vol_10d
high_low_range_over_close
close_over_10dma_minus_1
close_over_20dma_minus_1
volume_over_20d_avg
stock_minus_index_return
```

**Demo tensor shape:**

```text
tabular_tokens: [num_samples, 20, 11]
```

Each 11-dimensional time-step feature vector is projected into the common fusion dimension of 16. The 20 projected tabular time-step tokens attend with the other modality tokens inside the fusion Transformer.

The repo also contains a standalone `TabularTransformer` baseline. Its default config is model dimension 64, 4 heads, 2 Transformer layers, feed-forward dimension 128, and mean pooling. The all-in-one fusion demo does not use that standalone model directly; it uses the fusion model projection path.

### Image modality

**Raw input:** generated candlestick chart PNGs for the same stock/date sample.

**Demo image encoder:** `ImageTransformer`.

| Parameter | Demo value |
|---|---:|
| Image size | `64 x 64` |
| Patch size | `16 x 16` |
| Patch tokens | `16` patches + `1` CLS token |
| Encoder model dimension | `16` |
| Attention heads | `4` |
| Transformer encoder layers | `1` |
| Feed-forward dimension | `32` |

**Demo output shape:**

```text
image_tokens: [num_samples, 16]
```

The image encoder returns one pooled CLS embedding per chart. That vector is then projected into the fusion model's shared 16-dimensional space.

### Text modality

**Raw input:** leakage-safe market-summary text records generated from real OHLCV-derived features.

**Cutoff rule:**

```text
event_date <= sample end_date
```

**Demo encoder:** deterministic lightweight text tokenization via stable hashing. This is CPU-friendly and avoids depending on a large pretrained language model in the demo.

**Demo output shape:**

```text
text_tokens: [num_samples, 16]
```

**Transformer layers before fusion:** none in the current demo. Text is converted to a 16-dimensional vector and then projected into the fusion Transformer. A future enhancement could replace this with FinBERT or a compact sentence embedding model.

### Knowledge graph modality

**Raw input:** lightweight graph/context records: stock, sector, peer relationship, recent returns, and high-volume events.

**Demo KG token fields:**

```text
peer_count
peer_avg_recent_return
sector_avg_recent_return
event flag(s), e.g. high_volume
```

**Demo output shape:** usually:

```text
kg_tokens: [num_samples, 4]
```

The exact KG width can change if event types change. The all-in-one notebook prints the actual `kg_tokens` shape after saving the NPZ.

**Transformer layers before fusion:** none in the current demo. KG context is a compact numeric feature vector, then projected into the fusion Transformer. A future enhancement could add graph embeddings or a graph neural network.

## 5. Fusion Transformer architecture

The fusion model receives available modality tokens and projects all of them to a shared hidden size.

| Parameter | Demo value |
|---|---:|
| Type | encoder-only `TransformerEncoder` |
| Decoder | not used |
| Common model dimension | `16` |
| Attention heads | `4` |
| Transformer encoder layers | `1` |
| Feed-forward dimension | `32` |
| Activation | `GELU` |
| Normalization style | `norm_first=True` |
| Pooling | `CLS` |

**Fusion sequence concept:**

```text
[CLS]
+ projected tabular tokens, shape [20, 16]
+ projected image token, shape [1, 16]
+ projected text token, shape [1, 16]
+ projected KG token, shape [1, 16]
-> TransformerEncoder self-attention
-> pooled fused representation
-> binary logit: outperformance vs Nifty
```

For the default all-modality demo, the combined sequence is approximately 24 tokens including CLS.

## 6. Output layer and nonlinearity

The output head is intentionally simple:

```text
pooled fused representation
  -> Dropout
  -> Linear(model_dim -> 1)
  -> binary logit
```

During training, the model should use the raw logit with:

```text
BCEWithLogitsLoss
```

That loss applies the sigmoid internally in a numerically stable way.

During inference, convert the logit into probability with:

```text
probability = sigmoid(logit)
```

So the final nonlinearity is effectively sigmoid, but it is **not** manually applied before `BCEWithLogitsLoss` during training.

In [ ]:
# Compact architecture summary for the demo.
demo_architecture = {
    'tabular': {
        'shape': '[num_samples, 20, 11]',
        'pre_fusion_transformer_layers': 0,
        'fusion_projection_dim': 16,
    },
    'image': {
        'image_size': 64,
        'patch_size': 16,
        'patch_tokens': 16,
        'encoder_model_dim': 16,
        'encoder_heads': 4,
        'encoder_layers': 1,
        'encoder_ff_dim': 32,
    },
    'text': {
        'shape': '[num_samples, 16]',
        'encoder': 'deterministic stable text token, no text Transformer in demo',
        'fusion_projection_dim': 16,
    },
    'kg': {
        'shape': 'usually [num_samples, 4], inspect NPZ for exact width',
        'encoder': 'compact numeric KG context token, no GNN in demo',
        'fusion_projection_dim': 16,
    },
    'fusion_transformer': {
        'type': 'encoder-only Transformer',
        'decoder': 'not used',
        'model_dim': 16,
        'num_heads': 4,
        'num_layers': 1,
        'ff_dim': 32,
        'activation': 'GELU',
        'pooling': 'CLS',
    },
    'learning': {
        'supervision': 'labelled samples: y = 1 if stock outperforms Nifty over horizon else 0',
        'self_attention_role': 'combines and reweights signals across tokens/modalities',
        'loss': 'BCEWithLogitsLoss(prediction_logit, true_label)',
        'update_rule': 'backpropagation + optimizer.step()',
    },
    'output_head': {
        'layers': 'Dropout -> Linear(model_dim -> 1)',
        'training_loss': 'BCEWithLogitsLoss(raw_logit, label)',
        'inference_probability': 'sigmoid(logit)',
    },
}
demo_architecture

## 7. Suggested narration

> The model is encoder-only. We are not generating text or sequences, so we do not need a Transformer decoder. We create supervised training samples where each row contains aligned tabular, image, text, and KG tokens plus a label saying whether the stock outperformed Nifty over the future horizon. Self-attention helps the model mix signals across tokens and modalities, but the learning signal comes from comparing the predicted logit with the actual label using binary cross-entropy loss. During training, `BCEWithLogitsLoss` consumes the raw logit and applies sigmoid internally. During inference, sigmoid converts the logit into the probability of outperformance versus the Nifty.